# Librerias

In [ ]:
# -------------------------------
# Librerías para la construcción del modelo (TensorFlow / Keras)
# -------------------------------
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import VGG16, MobileNetV2, ResNet101V2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import image_dataset_from_directory, load_img, img_to_array
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2


# -------------------------------
# Librerías para preprocesamiento, métricas y utilidades
# -------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import label_binarize
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from pprint import pformat
import psutil
import GPUtil
import warnings
import time
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# Uso GPU

In [ ]:
# Verificar versión de TensorFlow y disponibilidad de GPU
print(f"Versión de TensorFlow: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("GPUs detectadas:")
    for gpu in gpus:
        print(f" - {gpu}")
    print("Aceleración por GPU activada:", tf.test.is_built_with_cuda())
else:
    print("No se encontró ninguna GPU disponible.")

# Ignorar advertencias para mantener la salida limpia
warnings.filterwarnings("ignore")


# Rutas de archivos

In [ ]:
# Ruta de Representaciones T-F
data_paths = {
    "Mel_512": "..\\..\\Analisis_Espectros\\DataSetImages_FULL\\DataSetImages\\Spectrogram\\Mel_512",
    "Mel_1024": "..\\..\\Analisis_Espectros\\DataSetImages_FULL\\DataSetImages\\Spectrogram\\Mel_1024",
    "Mel_2048": "..\\..\\Analisis_Espectros\\DataSetImages_FULL\\DataSetImages\\Spectrogram\\Mel_2048",
    "Morlet": "..\\..\\Analisis_Espectros\\DataSetImages_FULL\\DataSetImages\\Scalogram\\Morlet",
    "Bump": "..\\..\\Analisis_Espectros\\DataSetImages_FULL\\DataSetImages\\Scalogram\\Bump"
}

In [ ]:
def save_plot(fig, filename):
    filepath = os.path.join(GRAPHICS_DIR, filename)
    fig.savefig(filepath)
    plt.close(fig)

# Carga y procesamiento

In [ ]:
def load_and_preprocess_data(data_path, batch_size=50, img_size=(224, 224), validation_split=0.3, seed=123):
    """
    Carga y preprocesa imágenes desde un directorio, dividiéndolas en conjuntos de entrenamiento y validación.

    Parámetros:
        data_path (str): Ruta al directorio que contiene las imágenes organizadas en subdirectorios por clase.
        batch_size (int): Número de imágenes por lote.
        img_size (tuple): Tamaño al que se redimensionarán las imágenes (ancho, alto).
        validation_split (float): Fracción de datos a utilizar para el conjunto de validación.
        seed (int): Semilla para garantizar la reproducibilidad en la división de datos.

    Retorna:
        train_ds (tf.data.Dataset): Conjunto de datos preprocesado para entrenamiento.
        val_ds (tf.data.Dataset): Conjunto de datos preprocesado para validación.
        class_names (list): Lista de nombres de las clases detectadas.
    """
    # Cargar el conjunto de imágenes para entrenamiento y validación
    raw_train_ds = tf.keras.utils.image_dataset_from_directory(
        data_path,
        validation_split=validation_split,
        subset="training",
        seed=seed,
        image_size=img_size,
        batch_size=batch_size
    )
    raw_val_ds = tf.keras.utils.image_dataset_from_directory(
        data_path,
        validation_split=validation_split,
        subset="validation",
        seed=seed,
        image_size=img_size,
        batch_size=batch_size
    )
    
    # Extraer los nombres de las clases
    class_names = raw_train_ds.class_names
    
    # Normalizar las imágenes al rango [0, 1]
    normalization_layer = tf.keras.layers.Rescaling(1.0 / 255)
    train_ds = raw_train_ds.map(lambda x, y: (normalization_layer(x), y))
    val_ds = raw_val_ds.map(lambda x, y: (normalization_layer(x), y))
    
    # Optimizar el rendimiento con cache y prefetch
    train_ds = train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
    val_ds = val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
    
    return train_ds, val_ds, class_names


# Creacion de modelos

In [ ]:
def create_modelResnet101V2(num_classes, input_shape=(224, 224, 3)):
    base_model = ResNet101V2(
        weights ="imagenet",    # Pesos preentrenados en ImageNet
        include_top=False,      # Excluir las capas densas superiores
        input_shape=input_shape # Tamano de entrada configurable
    )
    # Congelar la base para evitar entrenarla de nuevo
    base_model.trainable = False

    # Construccion del modelo secuencial con capas personalizadas
    model = tf.keras.models.Sequential([
        base_model,
        tf.keras.layers.GlobalAveragePooling2D(),   # GlobalAveragePooling2D reduce la dimensionalidad de forma eficiente
        tf.keras.layers.Dense(256, activation="relu", kernel_regularizer=l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(num_classes, activation="softmax") # Capa de salida con softmax pra clasificacion multiclase
    ])
    
    return model

# Transfer learning

In [ ]:
def transfer_learn_model(model, train_ds, val_ds, epochs, base_learning_rate,
                         cpu_usages, memory_usages, gpu_usages,callbacks=None):
    """
    Entrena un modelo utilizando transfer learning.

    El modelo se compila con el optimizador Adam y se utilizan callbacks para ajustar la tasa de aprendizaje,
    detener el entrenamiento temprano y monitorear el uso de recursos del sistema durante el entrenamiento.

    Parámetros:
        model (tf.keras.Model): Modelo a entrenar.
        train_ds (tf.data.Dataset): Conjunto de datos para entrenamiento.
        val_ds (tf.data.Dataset): Conjunto de datos para validación.
        epochs (int, opcional): Número de épocas para entrenar. Valor por defecto es 15.
        callbacks (list, opcional): Callbacks adicionales que se quieran incluir.
        cpu_usages (list, opcional): Lista para registrar el uso de CPU.
        memory_usages (list, opcional): Lista para registrar el uso de memoria.
        gpu_usages (list, opcional): Lista para registrar el uso de GPU.

    Retorna:
        history: Objeto History de Keras con los detalles del entrenamiento.
    """
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=base_learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"]
    )

    # Definir callbacks predeterminados
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=4, min_lr=1e-8
    )
    system_load_callback = SystemLoadCallback(cpu_usages, memory_usages, gpu_usages)
    early_stopping_callback = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=2, restore_best_weights=True
    )

    callbacks = [system_load_callback, early_stopping_callback, reduce_lr]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks
    )
    return history

# Fine-Tuning

In [ ]:
def fine_tune_model(model, train_ds, val_ds, base_learning_rate, fine_tune_at, epochs,
                    cpu_usages, memory_usages, gpu_usages, callbacks=None):
    """
    Realiza ajuste fino (fine-tuning) del modelo descongelando la base preentrenada y entrenando
    las capas superiores con una tasa de aprendizaje reducida.

    Se asume que la primera capa del modelo es la base preentrenada (por ejemplo, VGG16 sin la parte superior).
    Las primeras 'fine_tune_at' capas de dicha base se congelan para evitar su actualización durante el entrenamiento.

    Parámetros:
        model (tf.keras.Model): Modelo a ajustar.
        train_ds (tf.data.Dataset): Conjunto de datos para entrenamiento.
        val_ds (tf.data.Dataset): Conjunto de datos para validación.
        base_learning_rate (float, opcional): Tasa de aprendizaje para el ajuste fino (default: 1e-6).
        fine_tune_at (int, opcional): Índice a partir del cual se descongelarán las capas de la base (default: 9).
        epochs (int, opcional): Número de épocas para el ajuste fino (default: 35).
        cpu_usages (list, opcional): Lista para registrar el uso de CPU.
        memory_usages (list, opcional): Lista para registrar el uso de memoria.
        gpu_usages (list, opcional): Lista para registrar el uso de GPU.
        callbacks (list, opcional): Callbacks adicionales a incluir durante el entrenamiento.

    Retorna:
        history: Objeto History de Keras con el historial del entrenamiento.
    """
    # Descongelar la base del modelo (asumida en la primera capa)
    base_model = model.layers[0]
    base_model.trainable = True

    # Congelar las primeras 'fine_tune_at' capas de la base
    for layer in base_model.layers[:fine_tune_at]:
        layer.trainable = False

    # Compilar el modelo para ajuste fino con la tasa de aprendizaje reducida
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=base_learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"]
    )

    # Crear callbacks predeterminados
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=10, min_lr=1e-7
    )
    system_load_callback = SystemLoadCallback(cpu_usages, memory_usages, gpu_usages)
    early_stopping_callback = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=30, restore_best_weights=True
    )
    callbacks = [system_load_callback, early_stopping_callback, reduce_lr]

    # Entrenamiento del modelo con ajuste fino
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks
    )
    
    return history


# Clase para recursos

In [ ]:
class SystemLoadCallback(tf.keras.callbacks.Callback):
    def __init__(self, cpu_usages, memory_usages, gpu_usages):
        self.cpu_usages = cpu_usages
        self.memory_usages = memory_usages
        self.gpu_usages = gpu_usages
        self.gpus_available = len(tf.config.list_physical_devices('GPU')) > 0

    def on_epoch_end(self, epoch, logs=None):
        # Medir uso de CPU y memoria
        cpu_usage = psutil.cpu_percent(interval=1)
        memory_usage = psutil.virtual_memory().percent
        # Medir uso de GPU si hay disponible
        gpu_usage = 0.0
        if self.gpus_available:
            try:
                gpus = GPUtil.getGPUs()
                if gpus:
                    gpu_usage = gpus[0].memoryUtil * 100  # Uso memoria GPUs
            except Exception as e:
                print(f"Error obteniendo uso de GPU: {str(e)}")
        # Almacenar valores en las listas
        self.cpu_usages.append(cpu_usage)
        self.memory_usages.append(memory_usage)
        self.gpu_usages.append(gpu_usage)
        print(f"Epoch {epoch + 1}: CPU: {cpu_usage:.2f}%, Memoria: {memory_usage:.2f}%, GPU Memory: {gpu_usage:.2f}%")


# Generar reporte y matriz

In [ ]:
def get_classification_metrics(model, val_ds, class_names):
    """
    Extrae las etiquetas verdaderas, las predicciones y las probabilidades a partir del conjunto
    de validación, y genera el reporte de clasificación.

    Parámetros:
        model (tf.keras.Model): Modelo entrenado.
        val_ds (tf.data.Dataset): Conjunto de datos de validación.
        class_names (list): Lista con los nombres de las clases.

    Retorna:
        y_true (np.array): Etiquetas verdaderas.
        y_pred (np.array): Predicciones (clase con mayor probabilidad).
        y_probs (np.array): Probabilidades predichas para cada clase.
        report_dict (dict): Reporte de clasificación en formato de diccionario.
    """

    y_true, y_pred, y_probs = [], [], []
    for images, labels in val_ds:
        preds = model.predict(images)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))
        y_probs.extend(preds)
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_probs = np.array(y_probs)
    
    report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

    # Calcular specificity por clase utilizando la matriz de confusión
    labels_numeric = list(range(len(class_names)))
    cm = confusion_matrix(y_true, y_pred, labels=labels_numeric)
    specificity_per_class = {}
    for i, label in enumerate(labels_numeric):
        # Verdaderos Negativos (TN): suma de todos los elementos excepto la fila i y la columna i
        TN = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
        # Falsos Positivos (FP): suma de la columna i, excluyendo la diagonal
        FP = cm[:, i].sum() - cm[i, i]
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
        specificity_per_class[class_names[i]] = specificity
        
    macro_specificity = np.mean(list(specificity_per_class.values()))
    
    # Agregar specificity al reporte
    report_dict["macro_specificity"] = macro_specificity
    report_dict["specificity_per_class"] = specificity_per_class
    
    return y_true, y_pred, y_probs, report_dict

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title="", save_confusion=True):
    """
    Genera y muestra la matriz de confusión a partir de las etiquetas verdaderas y las predicciones.
    
    Parámetros:
        y_true (array-like): Etiquetas verdaderas.
        y_pred (array-like): Predicciones.
        class_names (list): Nombres de las clases.
        title (str, opcional): Título para la gráfica.
        save_confusion (bool, opcional): Si es True, guarda la imagen.
    """
    conf_matrix = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(11, 11))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap="viridis",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"{title} - Matriz de Confusión")
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Etiqueta Verdadera")
    plt.tight_layout()
    plt.show()

    if save_confusion:
        try:
            save_plot(fig, f"conf_matrix_{title}.png")
        except NameError:
            fig.savefig(f"conf_matrix_{title}.png")
        print(f"Matriz de confusión guardada como 'conf_matrix_{title}.png'.")

# Generar curva ROC

In [ ]:
def plot_roc_curve(y_true, y_probs, class_names, title, save_path=None, show=True):
    """
    Genera y muestra la curva ROC para cada clase.
    
    Parámetros:
        y_true (array-like): Etiquetas verdaderas.
        y_probs (array-like): Probabilidades predichas para cada clase.
        class_names (list): Nombres de las clases.
        title (str): Título para la gráfica.
        save_path (str, opcional): Ruta para guardar la imagen.
        show (bool, opcional): Si True, muestra la gráfica.
        
    Retorna:
        fig, ax: Figura y ejes de la gráfica.
    """
    from sklearn.metrics import roc_curve, auc
    import matplotlib.pyplot as plt
    from sklearn.preprocessing import label_binarize
    import numpy as np

    y_true_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    y_probs = np.array(y_probs)

    fig, ax = plt.subplots(figsize=(8, 6))
    
    for i, class_name in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
        auc_score = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"{class_name} (AUC = {auc_score:.2f})")
    
    ax.plot([0, 1], [0, 1], "k--", label="Referencia")
    ax.set_xlabel("Tasa de Falsos Positivos")
    ax.set_ylabel("Tasa de Verdaderos Positivos")
    ax.set_title(f"Curva ROC - {title}")
    ax.legend(loc="lower right")
    ax.grid(True)
    fig.tight_layout()

    if save_path:
        try:
            save_plot(fig, save_path)
        except NameError:
            fig.savefig(save_path)
        print(f"Figura guardada en: {save_path}")

    if show:
        plt.show()

    return fig, ax

# Grafico para metricas

In [ ]:
def plot_resume_metrics(df_results):
    """
    Genera gráficos de columnas agrupadas para cada métrica (Accuracy, Macro F1 y Weighted F1),
    donde el eje X representa las etapas ("Transfer Learning" y "Fine-Tuning")
    y la leyenda muestra las representaciones.

    Parámetros:
        df_results (pd.DataFrame): DataFrame que contiene los resultados y debe tener las columnas:
            - "Representation"
            - "Stage"
            - "Accuracy"
            - "Macro F1"
            - "Weighted F1"
    """
    import matplotlib.pyplot as plt

    metrics = ["Accuracy","Macro Recall", "Macro F1", "Weighted F1", "Macro Specificity"] 

    # Para cada métrica, se genera el gráfico de columnas agrupadas
    for metric in metrics:
        # Pivotear el DataFrame: filas = Stage, columnas = Representation, valores = métrica
        pivot_df = df_results.pivot(index="Stage", columns="Representation", values=metric)
        # Reordenar las filas si se requiere un orden específico en el eje X
        order = ["Transfer Learning", "Fine-Tuning"]
        pivot_df = pivot_df.reindex(order)

        fig, ax = plt.subplots(figsize=(12, 6))
        pivot_df.plot(kind="bar", ax=ax, width=0.8)

        ax.set_title(f"Comparación de {metric} por Representación y Etapa")
        ax.set_xlabel("Etapa")
        ax.set_ylabel(metric)
        plt.xticks(rotation=0)
        plt.legend(title="Representación", bbox_to_anchor=(1, 1))
        plt.tight_layout()
        plt.show()

# Graficas de uso de gpu

In [ ]:
def plot_system_load(cpu_usages, memory_usages, gpu_usages, title, save_path=None):

    # Definir rango de épocas
    epochs = list(range(1, len(cpu_usages) + 1))
    
    # Crear subplots: 1 fila x 3 columnas
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=["Uso de CPU por época", "Uso de Memoria por época", "Uso de GPU por época"]
    )
    
    # Agregar traza para CPU
    fig.add_trace(
        go.Scatter(x=epochs, y=cpu_usages, mode="lines+markers", name="Uso de CPU"),
        row=1, col=1
    )
    
    # Agregar traza para Memoria
    fig.add_trace(
        go.Scatter(x=epochs, y=memory_usages, mode="lines+markers", name="Uso de Memoria", line=dict(color="red")),
        row=1, col=2
    )
    
    # Agregar traza para GPU
    fig.add_trace(
        go.Scatter(x=epochs, y=gpu_usages, mode="lines+markers", name="Uso de GPU", line=dict(color="green")),
        row=1, col=3
    )
    
    # Actualizar layout para quitar las líneas de grid
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=False)
    
    # Configurar título y tamaño general de la figura
    fig.update_layout(
        title_text=title,
        height=500,
        width=1000,
        showlegend=False
    )
    
    # Guardar la gráfica si se especifica una ruta
    if save_path:
        fig.write_html(save_path)
        print(f"Gráfica guardada en: {save_path}")
    
    fig.show()

In [ ]:
def plot_resume_system_load(computational_load, training_times):

    # Extraer representaciones
    representations = [entry["Representation"] for entry in computational_load]
    n_reps = len(representations)
    
    # Etapas: Transfer Learning y Fine-Tuning
    stages = np.array(["Transfer Learning", "Fine-Tuning"])
    x = np.arange(len(stages))  # posiciones en el eje X para las etapas
    
    # Ancho total para el grupo de barras en cada etapa y ancho individual
    total_width = 0.8
    bar_width = total_width / n_reps

    # Extraer datos para cada métrica
    cpu_tl = np.array([entry["CPU_TL (avg)"] for entry in computational_load])
    cpu_ft = np.array([entry["CPU_FT (avg)"] for entry in computational_load])
    
    memory_tl = np.array([entry["Memory_TL (avg)"] for entry in computational_load])
    memory_ft = np.array([entry["Memory_FT (avg)"] for entry in computational_load])
    
    times_tl = np.array([entry["Time TL (seconds)"] for entry in training_times])
    times_ft = np.array([entry["Time FT (seconds)"] for entry in training_times])
    
    # Crear subplots para CPU, Memoria y Tiempos
    fig, axs = plt.subplots(1, 3, figsize=(20, 6))
    
    # Función auxiliar para dibujar las barras de cada representación en cada subplot
    def plot_grouped_bars(ax, values_tl, values_ft, ylabel, title):
        for i in range(n_reps):
            # Calcular posiciones para la representación i dentro de cada grupo
            positions = x - total_width/2 + (i + 0.5)*bar_width
            # Valores para las dos etapas para la representación i
            vals = [values_tl[i], values_ft[i]]
            ax.bar(positions, vals, width=bar_width, label=representations[i])
        ax.set_xticks(x)
        ax.set_xticklabels(stages)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.legend(title="Representaciones")
        ax.grid(True, axis='y', alpha=0.7)
    
    # Gráfico de Uso de CPU
    plot_grouped_bars(axs[0], cpu_tl, cpu_ft, "Uso de CPU (%)", "Uso de CPU")
    
    # Gráfico de Uso de Memoria
    plot_grouped_bars(axs[1], memory_tl, memory_ft, "Uso de Memoria (%)", "Uso de Memoria")
    
    # Gráfico de Tiempos de Entrenamiento
    plot_grouped_bars(axs[2], times_tl, times_ft, "Tiempo (segundos)", "Tiempo de Entrenamiento")
    
    plt.tight_layout()
    plt.show()

# Graficas de validación

In [ ]:
def plot_training_history(history, title="", save_path=None):
    """
    Grafica las métricas de entrenamiento (accuracy y loss) para los conjuntos de entrenamiento y validación.

    Parámetros:
        history (tf.keras.callbacks.History): Objeto History de Keras que contiene las métricas de entrenamiento.
        title (str, opcional): Título descriptivo para las gráficas.
        save_path (str, opcional): Ruta completa para guardar la figura. Si no se especifica, se guarda con un nombre predeterminado.

    Retorna:
        None
    """
    # Extraer métricas de entrenamiento y validación
    acc = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs = range(1, len(acc) + 1)

    # Crear figura con dos subplots: Accuracy y Loss
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))

    # Subplot para Accuracy
    axs[0].plot(epochs, acc, color="#1f77b4", linestyle='-', linewidth=2, label='Training Accuracy')
    axs[0].plot(epochs, val_acc, color="#ff7f0e", linestyle='--', linewidth=2, label='Validation Accuracy')
    axs[0].set_title(f"{title} - Accuracy")
    axs[0].set_xlabel("Épocas")
    axs[0].set_ylabel("Accuracy")
    axs[0].legend()
    axs[0].grid(True, alpha=0.3)

    # Subplot para Loss
    axs[1].plot(epochs, loss, color="#2ca02c", linestyle='-', linewidth=2, label='Training Loss')
    axs[1].plot(epochs, val_loss, color="#d62728", linestyle='--', linewidth=2, label='Validation Loss')
    axs[1].set_title(f"{title} - Loss")
    axs[1].set_xlabel("Épocas")
    axs[1].set_ylabel("Loss")
    axs[1].legend()
    axs[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Guardar la figura, utilizando save_path si se proporciona
    final_path = save_path if save_path else f"training_history_{title}.png"
    try:
        save_plot(fig, final_path)
    except NameError:
        fig.savefig(final_path)
    print(f"Figura guardada en: {final_path}")

# Consolidado y mostrar resultados

In [ ]:
def consolidate_and_show_results(final_results):
    """
    Crea un DataFrame consolidado a partir de los resultados finales, calcula una puntuación combinada
    promediando las métricas 'Accuracy', 'Macro F1' y 'Weighted F1', y determina la mejor representación
    basada en este puntaje promedio.

    Parámetros:
        final_results (list): Lista de diccionarios con los resultados por representación y etapa.

    Retorna:
        df_results (pd.DataFrame): DataFrame consolidado con la columna "Average Score".
        best_model (pd.Series): La fila del DataFrame que representa el mejor modelo basado en el puntaje combinado.
    """
    # Crear DataFrame consolidado a partir de los resultados finales
    df_results = pd.DataFrame(final_results)
    df_results_final = df_results.drop(columns=["Specificity Per Class"])
    print("\n**Reporte Consolidado:**\n")
    print(df_results_final)

    # Validar que existan las columnas necesarias para el cálculo
    required_metrics = ["Accuracy","Macro Recall", "Macro F1", "Weighted F1","Macro Specificity"]   # Nuevo: promedio de specificity
    missing = [metric for metric in required_metrics if metric not in df_results_final.columns]
    if missing:
        raise KeyError(f"Faltan las siguientes columnas en el DataFrame: {missing}")

    # Calcular puntaje combinado (promedio de las métricas especificadas)
    df_results_final["Average Score"] = df_results_final[required_metrics].mean(axis=1).round(4)

    # Determinar el mejor modelo basado en el puntaje combinado
    best_model = df_results_final.sort_values(by="Average Score", ascending=False).iloc[0]

    print("\n**Mejor Representación (Basado en Todas las Métricas):**")
    print(best_model)
    
    return df_results_final, best_model



# PIPELINE Resnet101V2

In [ ]:
def run_experiment(data_path, representation):
    """
    Ejecuta el experimento de entrenamiento para una sola representación.
    
    Este proceso realiza las siguientes etapas:
      - Carga y preprocesa los datos.
      - Crea el modelo base.
      - Entrena el modelo en dos fases: Transfer Learning y Fine-Tuning.
      - Extrae las métricas de clasificación (y_true, y_pred, y_probs) para cada etapa.
      - Retorna las métricas de clasificación, tiempos de entrenamiento, carga computacional promedio,
        los objetos history de cada etapa y los datos de métricas (incluyendo las series de uso de recursos).
    
    Parámetros:
        data_path (str): Ruta a los datos correspondientes a la representación.
        representation (str): Nombre de la representación.
    
    Retorna:
        final_results (list of dict): Resultados de clasificación para Transfer Learning y Fine-Tuning.
        training_time (dict): Tiempos de entrenamiento (en segundos) para cada etapa.
        computational_load (dict): Promedio de carga computacional (CPU y Memoria) para cada etapa.
        histories (dict): Diccionario con los objetos history de cada etapa, con claves:
            "Transfer Learning" y "Fine-Tuning".
        metrics_all (dict): Diccionario con los datos extraídos de clasificación para cada etapa, incluyendo las series de uso.
            Contiene las claves "Transfer Learning" y "Fine-Tuning", cada una con:
                - y_true
                - y_pred
                - y_probs
                - report (reporte de clasificación)
                - cpu_usage_series
                - memory_usage_series
                - gpu_usage_series
    """

    # Cargar y preprocesar datos
    train_ds, val_ds, class_names = load_and_preprocess_data(data_path)
    num_classes = len(class_names)

    # Crear modelo base para la representación actual
    #----------------------- NOTA: CAMBIAR MODELO DE PRUEBA-------------------------------
    model = create_modelResnet101V2(num_classes)
    #-------------------------------------------------------------------------------------
    
    # Inicializar listas para capturar la carga computacional ( los callbacks lo llenan)
    cpu_usages_tl, memory_usages_tl, gpu_usages_tl = [], [], []
    cpu_usages_ft, memory_usages_ft, gpu_usages_ft = [], [], []
    
    # -----------------------------
    # ** TRANSFER LEARNING **
    # -----------------------------
    start_time_tl = time.time()
    # Entrenar el modelo en la etapa de Transfer Learning
    history_tl = transfer_learn_model(
    model, 
    train_ds, 
    val_ds, 
    epochs=50, 
    base_learning_rate=1e-5,
    cpu_usages=cpu_usages_tl,
    memory_usages=memory_usages_tl,
    gpu_usages=gpu_usages_tl
    )

    total_time_tl = time.time() - start_time_tl

    # Extraer métricas de clasificación para Transfer Learning
    y_true_tl, y_pred_tl, y_probs_tl, report_tl = get_classification_metrics(model, val_ds, class_names)
    
    result_tl = {
        "Representation": representation,
        "Stage": "Transfer Learning",
        "Accuracy": report_tl["accuracy"],
        "Macro Recall": report_tl["macro avg"]["recall"],
        "Macro F1": report_tl["macro avg"]["f1-score"],
        "Weighted F1": report_tl["weighted avg"]["f1-score"],
        "Macro Specificity": report_tl["macro_specificity"],          # Nuevo: promedio de specificity
        "Specificity Per Class": report_tl["specificity_per_class"]
    }
    
    metrics_tl = {
        "y_true": y_true_tl,
        "y_pred": y_pred_tl,
        "y_probs": y_probs_tl,
        "report": report_tl
    }
    # Tiempo de espera 3 min, descanso de recursos
    time.sleep(150)
    # -----------------------------
    # ** FINE-TUNING **
    # -----------------------------
    start_time_ft = time.time()
    # Entrenar el modelo en la etapa de Fine-Tuning
    history_ft = fine_tune_model(
    model, 
    train_ds, 
    val_ds, 
    base_learning_rate=1e-6, 
    fine_tune_at=365, 
    epochs=50,
    cpu_usages=cpu_usages_ft,
    memory_usages=memory_usages_ft,
    gpu_usages=gpu_usages_ft
    )

    total_time_ft = time.time() - start_time_ft

    # Extraer métricas de clasificación para Fine-Tuning
    y_true_ft, y_pred_ft, y_probs_ft, report_ft = get_classification_metrics(model, val_ds, class_names)
    
    result_ft = {
        "Representation": representation,
        "Stage": "Fine-Tuning",
        "Accuracy": report_ft["accuracy"],
        "Macro Recall": report_ft["macro avg"]["recall"],
        "Macro F1": report_ft["macro avg"]["f1-score"],
        "Weighted F1": report_ft["weighted avg"]["f1-score"],
        "Macro Specificity": report_ft["macro_specificity"],          # Nuevo: promedio de specificity
        "Specificity Per Class": report_ft["specificity_per_class"]
    }
    
    metrics_ft = {
        "y_true": y_true_ft,
        "y_pred": y_pred_ft,
        "y_probs": y_probs_ft,
        "report": report_ft
    }
    
    # Guardar el modelo entrenado
    model.save(f"model_{representation}.keras")
    
    # Consolidar resultados
    final_results = [result_tl, result_ft]
    
    training_time = {
        "Representation": representation,
        "Time TL (seconds)": total_time_tl,
        "Time FT (seconds)": total_time_ft
    }
    
    computational_load = {
        "Representation": representation,
        "CPU_TL (avg)": sum(cpu_usages_tl) / len(cpu_usages_tl) if cpu_usages_tl else 0,
        "Memory_TL (avg)": sum(memory_usages_tl) / len(memory_usages_tl) if memory_usages_tl else 0,
        "CPU_FT (avg)": sum(cpu_usages_ft) / len(cpu_usages_ft) if cpu_usages_ft else 0,
        "Memory_FT (avg)": sum(memory_usages_ft) / len(memory_usages_ft) if memory_usages_ft else 0,
    }
    
    # Agregar las series de uso de recursos a las métricas extraídas de cada etapa
    metrics_tl["cpu_usage_series"] = cpu_usages_tl
    metrics_tl["memory_usage_series"] = memory_usages_tl
    metrics_tl["gpu_usage_series"] = gpu_usages_tl

    metrics_ft["cpu_usage_series"] = cpu_usages_ft
    metrics_ft["memory_usage_series"] = memory_usages_ft
    metrics_ft["gpu_usage_series"] = gpu_usages_ft

    # Retornar también los objetos history en un diccionario
    histories = {
        "Transfer Learning": history_tl,
        "Fine-Tuning": history_ft
    }
    
    # Consolidar las métricas extraídas
    metrics_all = {
        "Transfer Learning": metrics_tl,
        "Fine-Tuning": metrics_ft,
        "class_names" : class_names
    }
    
    return final_results, training_time, computational_load, histories, metrics_all

In [ ]:
all_final_results = []
all_training_times = []
all_computational_load = []
all_histories = {}   # Diccionario para almacenar histories por representación
all_metrics = {}     # Diccionario para almacenar métricas extraídas por representación, incluyendo uso de recursos

for representation, data_path in data_paths.items():
    final_results, training_time, computational_load, histories, metrics_all = run_experiment(data_path, representation)
    
    all_final_results.extend(final_results)
    all_training_times.append(training_time)
    all_computational_load.append(computational_load)
    all_histories[representation] = histories          # Guardamos el history de esta representación
    all_metrics[representation] = metrics_all            # Guardamos las métricas extraídas (clasificación + uso de recursos)
    
    print(f"Experimento para {representation} completado. Esperando 60 segundos para normalizar recursos...")
    time.sleep(150)

In [ ]:
for representation, histories in all_histories.items():
    # Graficar para Transfer Learning
    plot_training_history(histories["Transfer Learning"], title=f"{representation} - Transfer Learning")
    # Graficar para Fine-Tuning
    plot_training_history(histories["Fine-Tuning"], title=f"{representation} - Fine-Tuning")

In [ ]:
for representation, metrics in all_metrics.items():  # Supongamos que has acumulado los metrics de cada experimento
    class_names = metrics.get("class_names")
    if not class_names:
        print(f"Class names no definido para {representation}")
        continue
    
    # Para Transfer Learning:
    y_true_tl = metrics["Transfer Learning"]["y_true"]
    y_probs_tl = metrics["Transfer Learning"]["y_probs"]
    plot_roc_curve(y_true_tl, y_probs_tl, class_names,
                   title=f"{representation} - ROC (Transfer Learning)",
                   save_path=f"roc_{representation}_tl.png")
    
    # Para Fine-Tuning:
    y_true_ft = metrics["Fine-Tuning"]["y_true"]
    y_probs_ft = metrics["Fine-Tuning"]["y_probs"]
    plot_roc_curve(y_true_ft, y_probs_ft, class_names,
                   title=f"{representation} - ROC (Fine-Tuning)",
                   save_path=f"roc_{representation}_ft.png")

In [ ]:
for representation, metrics in all_metrics.items():
    class_names = metrics.get("class_names")
    # Para la etapa de Transfer Learning
    y_true_tl = metrics["Transfer Learning"]["y_true"]
    y_pred_tl = metrics["Transfer Learning"]["y_pred"]
    title_tl = f"{representation} - Transfer Learning"
    plot_confusion_matrix(y_true_tl, y_pred_tl, class_names, title=title_tl, save_confusion=True)
    
    # Para la etapa de Fine-Tuning
    y_true_ft = metrics["Fine-Tuning"]["y_true"]
    y_pred_ft = metrics["Fine-Tuning"]["y_pred"]
    title_ft = f"{representation} - Fine-Tuning"
    plot_confusion_matrix(y_true_ft, y_pred_ft, class_names, title=title_ft, save_confusion=True)

In [ ]:
# Convertir la lista de resultados a un DataFrame
df_results = pd.DataFrame(all_final_results)
display(df_results)
# Llamar a la función para generar las gráficas
plot_resume_metrics(df_results)

In [ ]:
for representation, metrics in all_metrics.items():
    # Para la etapa de Transfer Learning:
    cpu_series_tl = metrics["Transfer Learning"].get("cpu_usage_series", [])
    memory_series_tl = metrics["Transfer Learning"].get("memory_usage_series", [])
    gpu_series_tl = metrics["Transfer Learning"].get("gpu_usage_series", [])
    
    if cpu_series_tl and memory_series_tl and gpu_series_tl:
        title_tl = f"{representation} - Transfer Learning - Uso de Recursos"
        plot_system_load(cpu_series_tl, memory_series_tl, gpu_series_tl, title=title_tl,
                         save_path=f"{representation}_system_load_TL.html")
    else:
        print(f"No se encontraron series de uso para {representation} en Transfer Learning")
    
    # Para la etapa de Fine-Tuning:
    cpu_series_ft = metrics["Fine-Tuning"].get("cpu_usage_series", [])
    memory_series_ft = metrics["Fine-Tuning"].get("memory_usage_series", [])
    gpu_series_ft = metrics["Fine-Tuning"].get("gpu_usage_series", [])
    
    if cpu_series_ft and memory_series_ft and gpu_series_ft:
        title_ft = f"{representation} - Fine-Tuning - Uso de Recursos"
        plot_system_load(cpu_series_ft, memory_series_ft, gpu_series_ft, title=title_ft,
                         save_path=f"{representation}_system_load_FT.html")
    else:
        print(f"No se encontraron series de uso para {representation} en Fine-Tuning")


In [ ]:
plot_resume_system_load(all_computational_load, all_training_times)

In [ ]:
df_results, best_model = consolidate_and_show_results(all_final_results)

In [ ]:
from datetime import datetime
from pprint import pformat

# Suponiendo que 'df_results' se creó a partir de 'all_final_results'
df_results_sorted = df_results.sort_values(by="Average Score", ascending=False)

print("\n**Tabla de Resultados Ordenada por Puntaje Promedio:**\n")
print(df_results_sorted)

# Guardar métricas y resultados en un archivo de texto con timestamp
with open("resultados.txt", "a+", encoding="utf-8") as archivo:
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    archivo.write(f"=== Resultados Finales - {timestamp} ===\n")
    archivo.write("Tabla de Resultados:\n")
    archivo.write(df_results_sorted.to_string(index=False))
    archivo.write("\n\n=== Resultados Finales (lista) ===\n")
    archivo.write(pformat(all_final_results, indent=4))
    archivo.write("\n\n=== Carga Computacional ===\n")
    archivo.write(pformat(all_computational_load, indent=4))
    archivo.write("\n\n=== Tiempos de Entrenamiento ===\n")
    archivo.write(pformat(all_training_times, indent=4))
    archivo.write("\n" + "=" * 80 + "\n")
